# 11. 제한된 병렬 처리와 결과 검증


## Goal

동시성 1·2에서 같은 입력 순서의 결과가 나오는지 검증하고, 작업 하나의 실패를 전체 상태로 전달한다. [교안 11-1](../../11-parallel-jobs/11-1-bounded-workers.md)과 연결된다.


## Setup

저장소의 기준 구현을 노트북에 포함했다. 외부 연결 없이 임시 파일 세 개만 처리한다. wc -l은 개행 수를 세므로 모든 샘플은 개행으로 끝난다.


In [ ]:
from pathlib import Path
import os
import shutil
import tempfile

lab_dir = Path(tempfile.mkdtemp(prefix="bash-book-11-"))
os.environ["BASH_LAB_DIR"] = str(lab_dir)
print(f"새 임시 실습 디렉터리: {lab_dir}")


## Steps

### 1. 기준 구현과 입력 준비

코드의 PID 배열, wait_batch, 입력 번호별 파일, 최종 병합을 찾아 설명한다.


In [ ]:
%%bash
set -euo pipefail
cat > "$BASH_LAB_DIR/run-workers.sh" <<'COURSE_WORKER'
#!/usr/bin/env bash
# Bounded batches, Bash 3.2+. stdout: input index and newline count, in input order.
set -o pipefail

usage() { printf 'usage: run-workers.sh JOBS FILE...\n' >&2; }
(( $# >= 2 )) || { usage; exit 2; }
[[ $1 =~ ^[1-8]$ ]] || { usage; exit 2; }
jobs=$1
shift
work_dir=$(mktemp -d) || exit 1
pids=()
cleanup() {
    local status=$? pid
    # Active entries are only children started by this script, not arbitrary PIDs.
    for pid in "${pids[@]}"; do
        kill -TERM "$pid" 2>/dev/null || :
    done
    for pid in "${pids[@]}"; do
        wait "$pid" 2>/dev/null || :
    done
    rm -rf -- "$work_dir"
    return "$status"
}
trap cleanup EXIT
trap 'exit 130' INT
trap 'exit 143' TERM

wait_batch() {
    local index status=0 pid
    for index in "${!pids[@]}"; do
        pid=${pids[$index]}
        wait "$pid" || status=1
        unset 'pids[index]'
    done
    pids=()
    return "$status"
}

index=0
failed=0
for input in "$@"; do
    index=$((index + 1))
    (
        [[ -f $input && -r $input ]] || exit 1
        wc -l < "$input" > "$work_dir/$index.txt"
    ) &
    pids+=("$!")
    if (( ${#pids[@]} >= jobs )); then
        wait_batch || failed=1
    fi
done
wait_batch || failed=1
if (( failed != 0 )); then
    printf 'one or more inputs failed; no merged result emitted\n' >&2
    exit 1
fi
for ((item=1; item<=index; item++)); do
    count=$(< "$work_dir/$item.txt")
    printf '%s\t%d\n' "$item" "$count"
done
COURSE_WORKER
printf 'a\nb\n' > "$BASH_LAB_DIR/two words.txt"
printf 'c\n' > "$BASH_LAB_DIR/one.txt"
: > "$BASH_LAB_DIR/empty.txt"
bash -n "$BASH_LAB_DIR/run-workers.sh"


### 2. 직렬과 병렬 비교

예상: 입력 1은 2, 입력 2는 1, 입력 3은 0이다. 두 실행의 출력이 정확히 같아야 한다.


In [ ]:
%%bash
set -euo pipefail
inputs=("$BASH_LAB_DIR/two words.txt" "$BASH_LAB_DIR/one.txt" "$BASH_LAB_DIR/empty.txt")
bash "$BASH_LAB_DIR/run-workers.sh" 1 "${inputs[@]}" > "$BASH_LAB_DIR/serial.tsv"
bash "$BASH_LAB_DIR/run-workers.sh" 2 "${inputs[@]}" > "$BASH_LAB_DIR/parallel.tsv"
printf '1\t2\n2\t1\n3\t0\n' > "$BASH_LAB_DIR/expected.tsv"
cmp "$BASH_LAB_DIR/serial.tsv" "$BASH_LAB_DIR/parallel.tsv"
cmp "$BASH_LAB_DIR/expected.tsv" "$BASH_LAB_DIR/parallel.tsv"
cat "$BASH_LAB_DIR/parallel.tsv"


### 3. 일부 입력 실패

예상: 상태 1, stdout은 비어 있음, stderr에는 실패 설명이 남음.


In [ ]:
%%bash
set -euo pipefail
status=0
bash "$BASH_LAB_DIR/run-workers.sh" 2 "$BASH_LAB_DIR/one.txt" "$BASH_LAB_DIR/missing" > "$BASH_LAB_DIR/out" 2> "$BASH_LAB_DIR/err" || status=$?
[[ $status == 1 && ! -s $BASH_LAB_DIR/out && -s $BASH_LAB_DIR/err ]]
printf 'failure propagated; partial report not emitted\n'


## Checks

- 직렬·병렬 출력이 정확히 같은가?
- 완료 순서 대신 입력 번호로 병합하는 이유를 설명할 수 있는가?
- 동시성 0을 지정하면 사용 오류 2인가?
- 실패한 작업이 있으면 부분 보고서를 최종 결과로 내보내지 않는가?


## Next Steps

12장의 두 프로젝트에 요구사항·정답·실패 검증·설명 자료를 적용해 최종 제출한다.


In [ ]:
import shutil
from pathlib import Path
import os

lab_dir = Path(os.environ["BASH_LAB_DIR"])
shutil.rmtree(lab_dir, ignore_errors=True)
print(f"정리 완료: {lab_dir}")
